<a href="https://colab.research.google.com/github/ritakimani9-lang/machinelearning/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ritakimani9-lang/machinelearning/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

LANE and TASK: RANKING : which content pages should be reviewed for refesh first. The label is_declining_label (impressions down 20% or more, last 30days vs the previous 30) . The week 4 baseline is scored with precision@K, so I keep that as the primary metric.

WHY LOGISTIC REGRESSION AND RANDOM FOREST: the course's method table maps "which first?" tasks to a classifier's probability, ranked and scored by precision@k. logistic regression is the readable starting point: about 30,000 rows, mostly numeric features, and coefficients i can inspect. Random foresr is te stronger second step. It handles nonlinear effects and skewed count features without scaling and needs little tuning. I put both in one table so the complex model has to earn it's place over the simple one.

WHY NOT OTHERS: gradient boosting overfits and leaks easily on a derived label with only 32 clients, and i'd need to justify the extra risk. A decision tree adds no ranking power over these two. clustering doesn't fit, because there is an observed label.

LEAKAGE CONTROL:
the label is exactly (impressions_Last_30d-impression_prev_30d) / impressions_prev_30d , so all *_last_30d, *_prev_30d and trend_* columns are excluded , as are content_id and client_id. preprocessing sits inside a pipeline and is fit on training folds only


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.




## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split design: grouped 5-fold cross-validation on client_id.

The leakage risk. The dataset has 30,000 pages but only 32 clients, and the largest client accounts for 23% of the rows. Pages from the same client share publishing and update habits, so a random split would put a client's pages in both train and test and reward the model for memorizing clients. A random stratified split inflated Random Forest precision@100 from 0.80 to 0.96 on this data.

The design. GroupKFold(5) keeps every page of a client on one side of the split. Each fold trains on about 24,000 pages and tests on about 6,000 pages from clients the model never saw.

Why it matches use. The decision is which pages to review first, including for clients the model hasn't seen. Testing on unseen clients mirrors that.

Same split for baseline and models. The baseline score is computed on the same held-out rows of every fold, and both use precision@K, so the comparison is like-for-like. The baseline has nothing to fit, so it cannot leak.

Other leakage controls.

Imputers and scalers live inside a Pipeline, so they are fit on training folds only.
Label-derived columns are excluded from the features.
No hyperparameters were tuned on the test folds; the Random Forest settings were fixed in advance.
Each content_id is unique, so there are no duplicate pages across folds.

Known limits. With 32 clients, fold results vary a lot: test base rates range from 0.38 to 0.65. Fold 0 holds only the largest client. GroupKFold is deterministic and doesn't stratify by class. I report mean ± std across folds and avoid overclaiming from any single fold.

What the split does not protect against
Same-window features. impressions_90d, clicks_90d and ctr cover a window that overlaps the label's period. Grouping by client can't fix that, and your self-check should name it.
Time. I saw only one snapshot in the columns, so I couldn't build a time-based split. Check your data contract to confirm there is no date dimension.
A cohort artifact. The model's favorite pages (age 144 days, updated 104 days ago) look like a batch created together. If that batch spans several clients, grouping by client won't separate it. Check whether it's a pipeline artifact.
Few clients. 32 clients is a small sample for claiming the model generalizes to new clients. Treat the gain over baseline as directional.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [2]:
# %% Cell 1: setup and load
import numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score

SEED = 42
URL = 'https://raw.githubusercontent.com/ritakimani9-lang/machinelearning/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(URL)
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# Baseline score, identical to w04
stale = df['freshness_tier'].isin(['91-180', '181+'])
vis = df['impression_tier'].isin(['moderate', 'good', 'excellent'])
df['baseline_score'] = (stale & vis) * df['impressions_90d']

# %% Cell 2: features (leaky columns excluded), models, grouped CV
NUM = ['search_volume','competition','cpc','word_count','impressions_90d','clicks_90d',
       'sessions_90d','engaged_sessions_90d','content_age_days','days_since_last_update',
       'ctr','avg_position','engagement_rate','scroll_rate','ai_traffic_pct',
       'days_with_impressions','days_with_sessions']
CAT = ['content_type', 'main_intent', 'competition_level']
LEAKY = ['trend_pct','trend_direction','impressions_last_30d','impressions_prev_30d',
         'clicks_last_30d','clicks_prev_30d','sessions_last_30d','sessions_prev_30d',
         'is_declining_label','content_id','client_id']
assert not set(NUM + CAT) & set(LEAKY), 'leaky column in features'

def prep(scale):
    return ColumnTransformer([
        ('n', make_pipeline(SimpleImputer(strategy='median'),
                            StandardScaler() if scale else 'passthrough'), NUM),
        ('c', make_pipeline(SimpleImputer(strategy='most_frequent'),
                            OneHotEncoder(handle_unknown='ignore')), CAT)])

models = {
    'LogReg': make_pipeline(prep(True), LogisticRegression(max_iter=2000)),
    'RandForest': make_pipeline(prep(False), RandomForestClassifier(
        300, min_samples_leaf=20, n_jobs=-1, random_state=SEED)),
}

def precision_at_k(s, y, k):
    return y[np.argsort(-s, kind='stable')[:k]].mean()

X, y, g = df[NUM + CAT], df['is_declining_label'].values, df['client_id'].values
Ks = [20, 50, 100, 200]
folds = list(GroupKFold(5).split(X, y, g))
rows = []
for f, (tr, te) in enumerate(folds):
    r = {'fold': f, 'base_rate': y[te].mean(), 'test_clients': len(set(g[te]))}
    for k in Ks:
        r[f'baseline@{k}'] = precision_at_k(df['baseline_score'].values[te], y[te], k)
    for name, m in models.items():
        m.fit(X.iloc[tr], y[tr])
        p = m.predict_proba(X.iloc[te])[:, 1]
        r[f'{name}_auc'] = roc_auc_score(y[te], p)
        for k in Ks:
            r[f'{name}@{k}'] = precision_at_k(p, y[te], k)
    rows.append(r)
res = pd.DataFrame(rows)
print(res.round(2).to_string())
print(res.drop(columns='fold').agg(['mean', 'std']).T.round(3))   # the comparison table

# %% Cell 3: error analysis (fold 0, Random Forest)
tr, te = folds[0]
m = models['RandForest']; m.fit(X.iloc[tr], y[tr])
test = df.iloc[te].copy(); test['p'] = m.predict_proba(X.iloc[te])[:, 1]
top = test.sort_values('p', ascending=False).head(100)
print('precision@100:', top['is_declining_label'].mean())
print(top['client_id'].value_counts())

pi = permutation_importance(m, X.iloc[te], y[te], scoring='roc_auc',
                            n_repeats=5, random_state=SEED, n_jobs=-1)
print(pd.Series(pi.importances_mean, index=NUM + CAT).sort_values(ascending=False).head(8).round(3))

cols = ['avg_position','content_age_days','ctr','impressions_90d','days_since_last_update','search_volume']
print(pd.DataFrame({'top100': top[cols].median(), 'all_test': test[cols].median()}))
print(top[top['is_declining_label'] == 0].head(3)[cols + ['trend_direction','trend_pct','p']])

   fold  base_rate  test_clients  baseline@20  baseline@50  baseline@100  baseline@200  LogReg_auc  LogReg@20  LogReg@50  LogReg@100  LogReg@200  RandForest_auc  RandForest@20  RandForest@50  RandForest@100  RandForest@200
0     0       0.49             1         0.45         0.32          0.34          0.40        0.60        0.8       0.72        0.72        0.74            0.65           0.85           0.84            0.84            0.83
1     1       0.65             7         0.55         0.48          0.51          0.48        0.59        0.6       0.68        0.67        0.69            0.63           0.85           0.84            0.86            0.86
2     2       0.38             8         0.25         0.28          0.27          0.40        0.65        0.8       0.74        0.79        0.76            0.74           0.95           0.90            0.85            0.78
3     3       0.62             8         0.60         0.70          0.76          0.83        0.65        0.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Base rate averages 0.54. The baseline's mean precision@100 is 0.51, so no better than picking at random, with a fold-to-fold std of 0.21. Logistic Regression reaches 0.75 and Random Forest 0.80 at K=100. Overall AUC is modest (about 0.63 for LogReg, 0.68 for RF). The models find a confident pocket at the top of the list but rank the middle of the list only weakly.

Where the models fail. Results are not uniform. In fold 3 the baseline beat both models at K=100 (0.76 vs 0.73 and 0.67), and RF was below base rate at K=20 (0.55 vs 0.62). Fold 0 holds only the largest client, so it tests a single client, and the other folds each hold 7 to 8. That variation is a reason to treat gains as directional, not proven.

What the model relies on. Permutation importance is dominated by avg_position and content_age_days. Decline rates support this: pages ranked at position 20+ decline far less (24 to 32%) than pages in the top 5 (61%), and newer pages (91 to 236 days old) decline more (60%) than older ones (about 41%).

What the top picks look like. In fold 0 the top 100 were all one cohort: content age 144 days, last updated 104 days ago, near-zero CTR, search volume 0, and missing word count. They are also small pages (median 653 impressions vs 1,692 for all test pages). The model therefore trades page size for decline likelihood, the opposite of the baseline's logic.

Three wrong picks. All three came from that same cohort. One was actually growing (+47.8%). One was stable (−2.4%). One was borderline (−15.4%, just short of the −20% cutoff). They look identical to the true decliners on the features the model sees.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.